In [ ]:
# ── CELL 0: Imports + Dataset Setup ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score, classification_report

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)

# Shared dataset
n = 1000
df = pd.DataFrame({
    'age':        np.random.randint(22, 60, n),
    'experience': np.random.randint(0, 35, n),
    'score':      np.random.normal(75, 10, n).clip(40, 100),
    'salary':     np.random.exponential(60000, n),
})
prob_promoted = 1 / (1 + np.exp(-(
    -5 + df['score'] * 0.05 + df['experience'] * 0.1
)))
df['promoted'] = (np.random.rand(n) < prob_promoted).astype(int)
print('Dataset shape:', df.shape)
print(df.head())

In [ ]:
# ── CELL 1: STEP 1 — Basic Probability ────────────────────────────────────────
# Classical probability — fair die
p_six     = 1 / 6
print(f'P(rolling 6) = {p_six:.4f}')

# Empirical probability — simulate
rolls     = np.random.randint(1, 7, size=100_000)
p_six_emp = np.mean(rolls == 6)
print(f'Empirical P(rolling 6) = {p_six_emp:.4f}  (converges to 0.1667)')

# Joint: P(6 twice) = P(6) x P(6)
p_two_sixes = p_six * p_six
print(f'P(6 twice)   = {p_two_sixes:.4f}')

# Union: P(5 or 6) — mutually exclusive
p_five_or_six = 1/6 + 1/6 - 0
print(f'P(5 or 6)    = {p_five_or_six:.4f}')

# Complement: P(NOT 6)
p_not_six = 1 - p_six
print(f'P(not 6)     = {p_not_six:.4f}')

In [ ]:
# ── CELL 2: STEP 2 — Conditional Probability P(A|B) ───────────────────────────
dept     = np.random.choice(['Eng', 'Sales', 'HR'], 10_000, p=[0.5, 0.3, 0.2])
promoted = np.zeros(10_000, dtype=int)
rates    = {'Eng': 0.40, 'Sales': 0.20, 'HR': 0.10}
for i, d in enumerate(dept):
    promoted[i] = np.random.rand() < rates[d]

emp = pd.DataFrame({'dept': dept, 'promoted': promoted})

# Unconditional
p_prom = emp['promoted'].mean()
print(f'P(promoted)         = {p_prom:.4f}')

# Conditional
p_prom_eng   = emp[emp['dept']=='Eng']['promoted'].mean()
p_prom_sales = emp[emp['dept']=='Sales']['promoted'].mean()
print(f'P(promoted | Eng)   = {p_prom_eng:.4f}  <- higher')
print(f'P(promoted | Sales) = {p_prom_sales:.4f}')

# Manual formula P(A|B) = P(A and B) / P(B)
p_prom_and_eng = ((emp['promoted']==1) & (emp['dept']=='Eng')).mean()
p_eng          = (emp['dept']=='Eng').mean()
print(f'Manual P(promoted|Eng) = {p_prom_and_eng / p_eng:.4f}  (same result)')

In [ ]:
# ── CELL 3: STEP 3 — Independence Check ───────────────────────────────────────
n = 100_000

# Independent: two coin flips
coin1 = np.random.randint(0, 2, n)
coin2 = np.random.randint(0, 2, n)

p_h1        = coin1.mean()
p_h2        = coin2.mean()
p_h1_and_h2 = ((coin1==1) & (coin2==1)).mean()

print('INDEPENDENT COINS:')
print(f'  P(H1)        = {p_h1:.4f}')
print(f'  P(H2)        = {p_h2:.4f}')
print(f'  P(H1 & H2)   = {p_h1_and_h2:.4f}')
print(f'  P(H1)*P(H2)  = {p_h1*p_h2:.4f}  <- equal = Independent')

# Dependent: experience drives salary
experience = np.random.randint(0, 30, n)
salary     = experience * 2000 + np.random.randint(30000, 50000, n)

p_high_sal             = (salary > 80000).mean()
p_high_sal_high_exp    = (salary[experience > 20] > 80000).mean()
print('\nDEPENDENT (experience & salary):')
print(f'  P(high salary)              = {p_high_sal:.4f}')
print(f'  P(high salary | high exp)   = {p_high_sal_high_exp:.4f}  <- different = Dependent')

In [ ]:
# ── CELL 4: STEP 4 — Bayes' Theorem: Spam Filter ──────────────────────────────
#
# P(spam)              = 0.40
# P("FREE" | spam)     = 0.80
# P("FREE" | legit)    = 0.10
# -> P(spam | "FREE") = ?

p_spam             = 0.40
p_legit            = 0.60
p_free_spam        = 0.80
p_free_legit       = 0.10

# Law of Total Probability
p_free             = p_free_spam * p_spam + p_free_legit * p_legit

# Bayes' Theorem
p_spam_given_free  = (p_free_spam * p_spam) / p_free

print("Bayes' Theorem — Spam Filter")
print(f'  Prior     P(spam)           = {p_spam:.2f}')
print(f'  Likelihood P(FREE|spam)     = {p_free_spam:.2f}')
print(f'  P(FREE)                     = {p_free:.4f}')
print(f'  Posterior P(spam|FREE)      = {p_spam_given_free:.4f}')
print(f'\n  Before seeing FREE: {p_spam*100:.0f}% spam')
print(f'  After  seeing FREE: {p_spam_given_free*100:.1f}% spam')

# Medical diagnosis variant: rare disease, accurate test
p_disease         = 0.01
p_pos_given_sick  = 0.95
p_pos_given_well  = 0.05
p_pos             = p_pos_given_sick * p_disease + p_pos_given_well * (1 - p_disease)
p_sick_given_pos  = (p_pos_given_sick * p_disease) / p_pos
print(f'\nMedical: P(disease | positive test) = {p_sick_given_pos:.4f}  ({p_sick_given_pos*100:.1f}%)')
print('  Counter-intuitive: only 16% chance of disease even with 95% accurate test!')

In [ ]:
# ── CELL 5: STEP 5 — Naive Bayes Classifier ───────────────────────────────────
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10,
                            n_informative=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred = gnb.predict(X_test)
y_prob = gnb.predict_proba(X_test)

print('Gaussian Naive Bayes:')
print(f'  Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'\n  First 5 probability outputs:')
for i in range(5):
    print(f'  Row {i}: P(0)={y_prob[i,0]:.3f}  P(1)={y_prob[i,1]:.3f}  -> predicted: {y_pred[i]}')

print(f'\n  Learned class priors: {gnb.class_prior_}')
print(f'  Feature means shape: {gnb.theta_.shape}  (2 classes x 10 features)')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# ── CELL 6: STEP 6 — Probability vs Likelihood (MLE) ──────────────────────────
from scipy.stats import norm

observed_data = np.array([2.1, 1.9, 2.3, 2.0, 2.2])
sigma         = 0.2

mus = np.linspace(1.5, 2.8, 300)

# Log-likelihood for each candidate mean
log_likelihoods = [
    np.sum(norm.logpdf(observed_data, loc=mu, scale=sigma))
    for mu in mus
]

best_mu  = mus[np.argmax(log_likelihoods)]
true_mle = observed_data.mean()

print(f'Data:              {observed_data}')
print(f'MLE (grid search): mu = {best_mu:.3f}')
print(f'MLE (formula):     mu = {true_mle:.3f}  (sample mean = MLE for Normal dist)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mus, log_likelihoods, 'steelblue', linewidth=2)
ax.axvline(best_mu, color='red', linestyle='--', label=f'MLE mu = {best_mu:.3f}')
ax.set_title('Log-Likelihood vs Candidate Mean\n(MLE = parameter that makes data most probable)')
ax.set_xlabel('Candidate mu')
ax.set_ylabel('Log-Likelihood')
ax.legend()
plt.tight_layout()
plt.show()

print('\nNeural network training = minimizing Negative Log-Likelihood = Cross-Entropy Loss')

In [ ]:
# ── CELL 7: STEP 7 — Log Probability and Cross-Entropy Loss ───────────────────
# Underflow problem
probabilities = np.full(500, 0.9)

product = 1.0
for p in probabilities:
    product *= p
print(f'Direct multiplication (500 probs): {product}')  # 0.0 = underflow

log_sum = np.sum(np.log(probabilities))
print(f'Log-probability sum:               {log_sum:.4f}')
print(f'Equivalent probability:            {np.exp(log_sum):.2e}')

# Cross-entropy loss
y_pred        = np.array([0.1, 0.7, 0.2])
y_true        = np.array([0,   1,   0  ])
cross_entropy = -np.sum(y_true * np.log(y_pred + 1e-9))
print(f'\nCross-entropy loss = {cross_entropy:.4f}')
print(f'  = -log(0.7) = {-np.log(0.7):.4f}')
print(f'  Penalizes being less confident in the correct class')

# Perplexity (LLM evaluation metric)
avg_nll    = -log_sum / len(probabilities)
perplexity = np.exp(avg_nll)
print(f'\nPerplexity: {perplexity:.4f}')
print('  Lower perplexity = better language model')

In [ ]:
# ── CELL 8: STEP 8 — Compare Naive Bayes vs Logistic Regression ───────────────
X = df[['age', 'experience', 'score', 'salary']].values
y = df['promoted'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

gnb = GaussianNB()
gnb.fit(X_train_sc, y_train)
probs_gnb = gnb.predict_proba(X_test_sc)[:, 1]

lr = LogisticRegression()
lr.fit(X_train_sc, y_train)
probs_lr = lr.predict_proba(X_test_sc)[:, 1]

print(f'Naive Bayes    -- Log Loss: {log_loss(y_test, probs_gnb):.4f}')
print(f'Logistic Reg.  -- Log Loss: {log_loss(y_test, probs_lr):.4f}')
print('Lower log-loss = better calibrated probabilities')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, probs, name in zip(axes,
                            [probs_gnb, probs_lr],
                            ['Naive Bayes', 'Logistic Regression']):
    ax.hist(probs[y_test == 0], bins=30, alpha=0.6, label='Not Promoted', color='steelblue')
    ax.hist(probs[y_test == 1], bins=30, alpha=0.6, label='Promoted',     color='tomato')
    ax.set_title(f'{name}\nP(promoted | features)')
    ax.set_xlabel('Predicted Probability')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 9: Practice — Fraud Detection with Bayes + Naive Bayes ───────────────
# Task 1: Bayes' Theorem manually
p_fraud         = 0.02
p_flag_fraud    = 0.90
p_flag_legit    = 0.05
p_flag          = p_flag_fraud * p_fraud + p_flag_legit * 0.98
p_fraud_flagged = (p_flag_fraud * p_fraud) / p_flag
print(f'P(fraud | flagged) = {p_fraud_flagged:.4f}  ({p_fraud_flagged*100:.1f}%)')
print('Even with 90% detection, only 27% of flagged = actually fraud (base rate is low)!')

# Tasks 2-4: Naive Bayes classifier
n_legit = 9800
n_fraud = 200

legit = np.column_stack([
    np.random.normal(50, 20, n_legit),
    np.random.randint(6, 22, n_legit),
    np.random.normal(3, 1, n_legit),
])
fraud = np.column_stack([
    np.random.normal(200, 80, n_fraud),
    np.random.choice([0,1,2,3,22,23], n_fraud),
    np.random.normal(8, 2, n_fraud),
])

Xf = np.vstack([legit, fraud])
yf = np.array([0]*n_legit + [1]*n_fraud)

Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2,
                                                stratify=yf, random_state=42)
gnb_fraud = GaussianNB()
gnb_fraud.fit(Xf_tr, yf_tr)
yf_prob = gnb_fraud.predict_proba(Xf_te)[:, 1]
yf_pred = gnb_fraud.predict(Xf_te)

print(f'\nFraud Detector:')
print(f'  Accuracy: {accuracy_score(yf_te, yf_pred):.4f}')
print(f'  Log-Loss: {log_loss(yf_te, yf_prob):.4f}')
print(classification_report(yf_te, yf_pred, target_names=['Legit', 'Fraud']))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(yf_prob[yf_te==0], bins=40, alpha=0.6, label='Legit',  color='steelblue')
ax.hist(yf_prob[yf_te==1], bins=40, alpha=0.6, label='Fraud',  color='tomato')
ax.set_title('P(fraud | features) -- two peaks near 0 and 1 = good model')
ax.set_xlabel('Predicted Probability of Fraud')
ax.legend()
plt.tight_layout()
plt.show()